# Module 3 Extended Lab: Advanced Movie Analytics with SQL

Welcome to the extended lab!

You already know how to connect to the database and run basic queries.  
In this lab you will go deeper: joining tables, aggregating ratings, ranking directors, analyzing engagement, and answering business questions that marketing and content teams actually care about.

**Database:** `mubi_movies_ratings.db`  
**Tables:** `movies` and `ratings` (same schema as the graded assignment)

### Learning Objectives
- Write multi-table joins
- Use `GROUP BY`, `HAVING`, and aggregate functions
- Rank results with window functions / `LIMIT` + ordering
- Combine SQL results with pandas for further analysis and visualization
- Answer real analytical questions about popularity, quality, and engagement

---

## Table of Contents
1. [Setup & Connection](#setup)
2. [Exercise 1 – Movies with the Most Ratings](#ex1)
3. [Exercise 2 – Highest Average User Ratings](#ex2)
4. [Exercise 3 – Directors Ranked by Average Rating](#ex3)
5. [Exercise 4 – Popularity vs Quality Analysis](#ex4)
6. [Exercise 5 – Most Engaging Critiques (with movie titles)](#ex5)
7. [Exercise 6 – Yearly Trends of High-Quality Movies](#ex6)
8. [Bonus Challenge – CTE + Window Function](#bonus)

<a id="setup"></a>
## 1. Setup & Connection

Run the cell below to import libraries and connect to the database.

In [ ]:
# 🔒 Locked cell – do not edit
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

db_path = "mubi_movies_ratings.db"
connection = sqlite3.connect(db_path)
print("Connected successfully!")

<a id="ex1"></a>
## Exercise 1 – Movies with the Most Ratings

**Business question:** Which titles have received the highest number of user ratings?  
These are the films with the strongest community engagement.

**Directions**
1. Join `movies` and `ratings` on `movie_id`
2. Count the number of ratings per movie
3. Return `movie_title`, `movie_release_year`, `movie_popularity`, and the rating count
4. Order by the number of ratings descending and limit to the top 15

In [ ]:
# GRADED CELL: Exercise 1

### START CODE HERE ###

query_most_ratings = """
SELECT 
    m.movie_title,
    m.movie_release_year,
    m.movie_popularity,
    COUNT(r.rating_id) AS num_ratings
FROM movies m
JOIN ratings r ON m.movie_id = r.movie_id
GROUP BY m.movie_id, m.movie_title, m.movie_release_year, m.movie_popularity
ORDER BY num_ratings DESC
LIMIT 15;
"""

most_ratings_df = pd.read_sql_query(query_most_ratings, connection)

### END CODE HERE ###

most_ratings_df

<a id="ex2"></a>
## Exercise 2 – Highest Average User Ratings

**Business question:** Which movies have the best average score from real users (not the pre-computed `rating` column)?

**Directions**
1. Join the two tables
2. Calculate the average of `rating_score`
3. Only keep movies that received **at least 50 ratings** (use `HAVING`)
4. Return title, year, average score, and number of ratings
5. Order by average score descending, limit to top 20

In [ ]:
# GRADED CELL: Exercise 2

### START CODE HERE ###

query_avg_ratings = """
SELECT 
    m.movie_title,
    m.movie_release_year,
    ROUND(AVG(r.rating_score), 2) AS avg_user_score,
    COUNT(*) AS num_ratings
FROM movies m
JOIN ratings r ON m.movie_id = r.movie_id
GROUP BY m.movie_id, m.movie_title, m.movie_release_year
HAVING COUNT(*) >= 50
ORDER BY avg_user_score DESC
LIMIT 20;
"""

avg_ratings_df = pd.read_sql_query(query_avg_ratings, connection)

### END CODE HERE ###

avg_ratings_df

<a id="ex3"></a>
## Exercise 3 – Directors Ranked by Average Rating

**Business question:** Which directors consistently deliver highly rated films?

**Directions**
1. Join `movies` and `ratings`
2. Group by `director_name`
3. Calculate average user score and total number of ratings
4. Only keep directors who have at least 5 movies and 100 total ratings
5. Order by average score descending and show the top 15

In [ ]:
# GRADED CELL: Exercise 3

### START CODE HERE ###

query_directors_ranked = """
SELECT 
    m.director_name,
    ROUND(AVG(r.rating_score), 2) AS avg_score,
    COUNT(DISTINCT m.movie_id) AS num_movies,
    COUNT(*) AS total_ratings
FROM movies m
JOIN ratings r ON m.movie_id = r.movie_id
WHERE m.director_name IS NOT NULL
GROUP BY m.director_name
HAVING COUNT(DISTINCT m.movie_id) >= 5 
   AND COUNT(*) >= 100
ORDER BY avg_score DESC
LIMIT 15;
"""

directors_ranked_df = pd.read_sql_query(query_directors_ranked, connection)

### END CODE HERE ###

directors_ranked_df

In [ ]:
# 🔒 Visualization cell
plt.figure(figsize=(10, 6))
sns.barplot(
    data=directors_ranked_df, 
    y="director_name", 
    x="avg_score", 
    palette="viridis",
    hue="director_name",
    legend=False
)
plt.title("Top Directors by Average User Score")
plt.xlabel("Average User Score")
plt.ylabel("Director")
plt.tight_layout()
plt.show()

<a id="ex4"></a>
## Exercise 4 – Popularity vs Quality Analysis

**Business question:** Are the most popular movies also the highest quality?

**Directions**
1. Create a query that returns, for every movie that has ratings:
   - `movie_title`
   - `movie_popularity`
   - average user score
   - number of ratings
2. Keep only movies with ≥ 30 ratings
3. Load the result into a DataFrame and create a scatter plot of popularity vs average score

In [ ]:
# GRADED CELL: Exercise 4

### START CODE HERE ###

query_pop_vs_quality = """
SELECT 
    m.movie_title,
    m.movie_popularity,
    ROUND(AVG(r.rating_score), 2) AS avg_score,
    COUNT(*) AS num_ratings
FROM movies m
JOIN ratings r ON m.movie_id = r.movie_id
GROUP BY m.movie_id, m.movie_title, m.movie_popularity
HAVING COUNT(*) >= 30
ORDER BY m.movie_popularity DESC;
"""

pop_quality_df = pd.read_sql_query(query_pop_vs_quality, connection)

### END CODE HERE ###

# Quick look
print(pop_quality_df.head(10))
print(f"\nTotal movies analyzed: {len(pop_quality_df)}")

In [ ]:
# 🔒 Visualization
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=pop_quality_df,
    x="movie_popularity",
    y="avg_score",
    size="num_ratings",
    sizes=(20, 200),
    alpha=0.6
)
plt.title("Movie Popularity vs Average User Score")
plt.xlabel("Mubi Popularity (number of users who love it)")
plt.ylabel("Average User Score (1-5)")
plt.tight_layout()
plt.show()

<a id="ex5"></a>
## Exercise 5 – Most Engaging Critiques (with movie titles)

**Business question:** Which critiques generated the most likes, and what movies were they about?

**Directions**
1. Join `ratings` with `movies`
2. Select movie title, the critique text, and `critique_likes`
3. Order by likes descending
4. Limit to the top 30 most-liked critiques

In [ ]:
# GRADED CELL: Exercise 5

### START CODE HERE ###

query_top_critiques = """
SELECT 
    m.movie_title,
    r.critique,
    r.critique_likes
FROM ratings r
JOIN movies m ON r.movie_id = m.movie_id
WHERE r.critique IS NOT NULL 
  AND r.critique != 'None'
ORDER BY r.critique_likes DESC
LIMIT 30;
"""

top_critiques_df = pd.read_sql_query(query_top_critiques, connection)

### END CODE HERE ###

print(f"Number of rows: {len(top_critiques_df)}\n")
top_critiques_df.head(10)

In [ ]:
# 🔒 Word cloud from the most-liked critiques
text = " ".join(top_critiques_df['critique'].dropna().str.lower().tolist())
wordcloud = WordCloud(width=900, height=450, background_color='white', 
                      colormap='plasma', max_words=80).generate(text)

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("Most Common Words in the 30 Most-Liked Critiques", fontsize=14)
plt.show()

<a id="ex6"></a>
## Exercise 6 – Yearly Trends of High-Quality Movies

**Business question:** How has the volume of highly-rated films changed over the decades?

**Directions**
1. Join the tables
2. Calculate average user score per movie
3. Keep only movies with average score ≥ 4.0 and at least 20 ratings
4. Group by release year and count how many such high-quality movies were released each year
5. Focus on years from 1970 onwards

In [ ]:
# GRADED CELL: Exercise 6

### START CODE HERE ###

query_yearly_quality = """
SELECT year, COUNT(*) AS high_quality_movies
FROM (
    SELECT 
        m.movie_release_year AS year,
        AVG(r.rating_score) AS avg_score
    FROM movies m
    JOIN ratings r ON m.movie_id = r.movie_id
    GROUP BY m.movie_id, m.movie_release_year
    HAVING AVG(r.rating_score) >= 4.0 
       AND COUNT(*) >= 20
       AND m.movie_release_year >= 1970
) AS high_quality
GROUP BY year
ORDER BY year;
"""

yearly_df = pd.read_sql_query(query_yearly_quality, connection)

### END CODE HERE ###

yearly_df.head(15)

In [ ]:
# 🔒 Visualization
plt.figure(figsize=(14, 6))
sns.lineplot(data=yearly_df, x="year", y="high_quality_movies", marker="o")
plt.title("Number of High-Quality Movies (≥ 4.0 avg score) Released per Year")
plt.xlabel("Release Year")
plt.ylabel("Count of High-Quality Movies")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<a id="bonus"></a>
## Bonus Challenge – CTE + Window Function

**Advanced task**

Write a single query that:
1. Calculates the average user score and number of ratings for every movie
2. Ranks movies within each decade by their average score
3. Returns only the **#1 ranked movie of each decade** (from 1950 onwards)

Use a CTE and the `RANK()` or `ROW_NUMBER()` window function.

In [ ]:
# GRADED CELL: Bonus (optional but highly recommended)

### START CODE HERE ###

query_decade_champions = """
WITH movie_scores AS (
    SELECT 
        m.movie_title,
        m.movie_release_year,
        (m.movie_release_year / 10) * 10 AS decade,
        ROUND(AVG(r.rating_score), 2) AS avg_score,
        COUNT(*) AS num_ratings
    FROM movies m
    JOIN ratings r ON m.movie_id = r.movie_id
    GROUP BY m.movie_id, m.movie_title, m.movie_release_year
    HAVING COUNT(*) >= 15
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY decade 
               ORDER BY avg_score DESC, num_ratings DESC
           ) AS rank_in_decade
    FROM movie_scores
    WHERE decade >= 1950
)
SELECT 
    decade,
    movie_title,
    movie_release_year,
    avg_score,
    num_ratings
FROM ranked
WHERE rank_in_decade = 1
ORDER BY decade;
"""

decade_champs_df = pd.read_sql_query(query_decade_champions, connection)

### END CODE HERE ###

decade_champs_df

### Final step – close the connection

In [ ]:
# 🔒 Locked cell
connection.close()
print("Connection closed. Lab complete!")

---

## Congratulations!

You have completed the extended lab.  
You now know how to:
- Join tables
- Aggregate and filter with `HAVING`
- Rank results
- Use CTEs and window functions
- Turn SQL results into meaningful business insights and visualizations

Feel free to experiment further (e.g., analyze subscriber vs non-subscriber ratings, critique length vs likes, etc.).

**Tips for success:**
- Always check the number of rows returned after filtering with `HAVING`.
- Use `ROUND(..., 2)` for clean average scores.
- When ranking with window functions, decide carefully between `RANK()`, `DENSE_RANK()`, and `ROW_NUMBER()`.
- Visualizations help stakeholders understand the story behind the numbers.